# Bank Marketing Prediction 

**Goal:** predict whether a bank customer will subscribe to a term deposit (`y`: yes / no) using supervised machine learning.

**Dataset:** UCI Bank Marketing, `bank-full.csv` (45,211 rows, 16 input columns + target `y`).
Citation: Moro, S., Rita, P., & Cortez, P. (2014). *Bank Marketing*. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306

**Steps:** load data → clean → preprocess → train 4 models → evaluate → compare.

## 1. Import libraries and load dataset

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

RANDOM_STATE = 42   # fixed seed so results are reproducible

In [4]:
df = pd.read_csv("bank-original.csv")   # the original file is only read, never changed

print("First five rows:")
display(df.head())
print("Shape (rows, columns):", df.shape)
print("Column names:", list(df.columns))

First five rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


Shape (rows, columns): (45211, 17)
Column names: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


## 2. Data cleaning

We only fix a problem if we actually find one. Every check below prints its result, and the summary at the end is built from those results.

In [6]:
# Check 1: missing values
missing_total = int(df.isna().sum().sum())

# Check 2: duplicate rows
duplicate_total = int(df.duplicated().sum())


In [ ]:
# Check 4: invalid values (rules based on what each column means)
invalid_checks = {
    "age not between 18 and 100":  ~df["age"].between(18, 100),
    "day not between 1 and 31":    ~df["day"].between(1, 31),
    "duration is negative":        df["duration"] < 0,
    "campaign is less than 1":     df["campaign"] < 1,
    "previous is negative":        df["previous"] < 0,
    "pdays below -1":              df["pdays"] < -1,
    "target not 'yes' or 'no'":    ~df["y"].isin(["yes", "no"]),
}
invalid_total = 0
for rule, mask in invalid_checks.items():
    n = int(mask.sum())
    invalid_total += n
    print(f"{rule:30s}: {n}")

# The text value "unknown" is a real category in this dataset, not an empty cell
print("\nColumns containing the category 'unknown':")
unknown_counts = (df == "unknown").sum()
print(unknown_counts[unknown_counts > 0].to_string())

age not between 18 and 100    : 0
day not between 1 and 31      : 0
duration is negative          : 0
campaign is less than 1       : 0
previous is negative          : 0
pdays below -1                : 0
target not 'yes' or 'no'      : 0

Columns containing the category 'unknown':
job            288
education     1857
contact      13020
poutcome     36959


In [9]:
# Apply cleaning ONLY where a problem was found
steps = []
if missing_total > 0:
    steps.append(f"{missing_total} missing cells found -> will be imputed in the pipeline (Section 3)")
else:
    steps.append("No missing values found -> nothing to fix")

if duplicate_total > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    steps.append(f"{duplicate_total} duplicate rows found -> removed")
else:
    steps.append("No duplicate rows found -> nothing removed")

if invalid_total > 0:
    steps.append(f"{invalid_total} invalid values found -> inspect before using the data")
else:
    steps.append("No invalid values found -> nothing to fix")

steps.append("'unknown' values kept as a normal category; outliers (e.g. large balances) kept because they are valid values")
steps.append("pdays = -1 means 'not previously contacted'; it is kept unchanged to preserve the original meaning")

print("\nDataset shape after cleaning:", df.shape)


Dataset shape after cleaning: (45211, 17)


## 3. Data preprocessing

* **Target:** `y` is converted to numbers (`yes = 1`, `no = 0`) and kept separate from the inputs.
* **`duration` is excluded.** It is the length of the last phone call, which is only known *after* the call. We want to predict *before* calling the customer, so using it would be **data leakage** (the model would use information from the future and look unrealistically good).
* **Numerical features:** fill missing values with the median, then standardise with `StandardScaler`.
* **Categorical features:** fill missing values with the most frequent value, then `OneHotEncoder`.
* **No leakage:** the preprocessing is placed *inside* each model's `Pipeline`, so it is fitted on the training data only and just applied to the test data.
* **Class imbalance:** most customers say "no", so every model uses `class_weight="balanced"` (mistakes on the rare "yes" class count more).

In [ ]:
# Separate target and inputs; drop duration (leakage)
y = df["y"].map({"yes": 1, "no": 0})
X = df.drop(columns=["y", "duration"])

# Separate numerical and categorical columns
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = [col for col in X.columns if col not in numeric_features]
print("Numerical features  :", numeric_features)
print("Categorical features:", categorical_features)

# Preprocessing steps for each column type

# if a numerical column has missing values, we replace them with its median.
numeric_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
# If a categorical column contains missing values, replace them with the most common category.
categorical_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_steps, numeric_features),
    ("cat", categorical_steps, categorical_features),
])

Numerical features  : ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [11]:
# Save the cleaned data (duration removed, y as 0/1) as a SEPARATE file; bank-full.csv is not touched.
# Scaling and one-hot encoding are NOT saved here: they happen inside the pipeline after the split.
processed = X.copy()
processed["y"] = y
processed.to_csv("processed_bank_marketing.csv", index=False)
print("Saved processed_bank_marketing.csv with shape", processed.shape)

Saved processed_bank_marketing.csv with shape (45211, 16)


In [12]:
# 80% training / 20% testing; stratify keeps the same yes/no ratio in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Training set:", X_train.shape, "| Test set:", X_test.shape)
print(f"Share of 'yes' -> all data: {y.mean():.3f} | train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

Training set: (36168, 15) | Test set: (9043, 15)
Share of 'yes' -> all data: 0.117 | train: 0.117 | test: 0.117


### 3.1 Outlier detection (IQR method)

An **outlier** is a value that is unusually far from most of the other values in the same column. We use the **Interquartile Range (IQR) method**:

* **Q1** = 25th percentile, **Q3** = 75th percentile (the middle 50% of the values lie between them)
* **IQR = Q3 − Q1**
* **Lower bound = Q1 − 1.5 × IQR**
* **Upper bound = Q3 + 1.5 × IQR**

Any value below the lower bound or above the upper bound is flagged as an outlier (this is also what the "dots" in a box plot show).

**Details of this step**
* The bounds are calculated on the **training data only** (`X_train`), so no information from the test set is used.
* We check the numerical features `age`, `balance`, `campaign`, `previous` and `pdays`. `day` is skipped because it is just the calendar day (1 to 31), so it cannot have extreme values. For `pdays`, the value `-1` means "not contacted before" (it is not a real number of days), so only customers who *were* contacted before are checked.
* This step only **detects and reports** outliers. The data is **not changed or deleted** here (see the decision below the results).

In [ ]:
# IQR outlier detection on the TRAINING data only; X_train itself is NOT modified
outlier_features = ["age", "balance", "campaign", "previous", "pdays"]

rows = []
checked_values = {}
for col in outlier_features:
    values = X_train[col]
    if col == "pdays":
        values = values[values != -1]          # -1 = "not previously contacted", not a real day count
    checked_values[col] = values

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1                              # IQR = Q3 - Q1
    lower_bound = q1 - 1.5 * iqr               # Lower bound = Q1 - 1.5 x IQR
    upper_bound = q3 + 1.5 * iqr               # Upper bound = Q3 + 1.5 x IQR
    n_outliers = int(((values < lower_bound) | (values > upper_bound)).sum())

    rows.append({
        "Feature": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "Lower bound": lower_bound, "Upper bound": upper_bound,
        "Values checked": len(values), "Outliers": n_outliers,
        "% outliers": 100 * n_outliers / len(values),
    })

outlier_table = pd.DataFrame(rows).set_index("Feature")
print("Number of IQR outliers per numerical feature (training data):")
display(outlier_table.round(2))

# Worked example so the calculation is easy to explain
ex = outlier_table.loc["balance"]
print(f"\nWorked example for 'balance':")
print(f"  IQR         = Q3 - Q1          = {ex['Q3']:.0f} - {ex['Q1']:.0f} = {ex['IQR']:.0f}")
print(f"  Lower bound = Q1 - 1.5 x IQR   = {ex['Q1']:.0f} - 1.5 x {ex['IQR']:.0f} = {ex['Lower bound']:.0f}")
print(f"  Upper bound = Q3 + 1.5 x IQR   = {ex['Q3']:.0f} + 1.5 x {ex['IQR']:.0f} = {ex['Upper bound']:.0f}")
print(f"  -> {int(ex['Outliers'])} balances ({ex['% outliers']:.1f}%) lie outside [{ex['Lower bound']:.0f}, {ex['Upper bound']:.0f}]")

# Box plots (one panel per feature because the scales are very different)
fig, axes = plt.subplots(1, len(outlier_features), figsize=(16, 4))
for ax, col in zip(axes, outlier_features):
    ax.boxplot(checked_values[col], flierprops=dict(marker="o", markersize=2, alpha=0.4))
    ax.set_title(f"{col}\n{int(outlier_table.loc[col, 'Outliers'])} outliers")
    ax.set_xticks([])
fig.suptitle("Box plots of numerical features (training data) - dots are IQR outliers")
plt.tight_layout()
plt.show()

**How to read the results (simple language)**

* The box in each plot is the middle 50% of customers. The "whiskers" show the normal range (the lower and upper bounds). Dots outside the whiskers are outliers.
* Check the table and plots above: features such as `balance` and `campaign` typically have a long tail on the high side (a few customers with a very large account balance or many contact attempts), and `age` may show a few unusually old customers. Compare the outlier counts and percentages to see which features are affected most.
* For `previous` (and partly `pdays`) most customers have the same value, so Q1 and Q3 are close together and the IQR is tiny. The bounds then become very tight and many ordinary values are flagged. This is a limitation of the IQR rule for features that are mostly constant, not a sign of bad data.

**Decision: the outliers are kept (no rows are deleted)**

* Earlier, the validity checks (Section 2) found no impossible values, so these are real customers, not errors. A very large balance or many contact attempts can be genuine and may even be related to subscribing.
* Deleting them would throw away real customers and change the class balance of the data.
* The tree-based models (Decision Tree, Random Forest, Gradient Boosting) split on thresholds, so extreme values do not distort them. Logistic Regression is more sensitive, but its inputs are standardised and the outliers are real, so we keep them and report this as a limitation.
* The original dataset and `X_train` are therefore left unchanged.

### 3.2 Feature selection (SelectKBest with mutual information)

**Why:** after one-hot encoding, the 15 input columns become about 50 model inputs. Many of them carry little information about whether a customer subscribes. Feature selection keeps only the most useful inputs.

**How it works**
* **Mutual information (MI)** measures how much knowing a feature reduces the uncertainty about the target `y`. MI = 0 means the feature tells us nothing about `y`; a larger value means a stronger link. Unlike correlation, it also detects non-linear relationships and works for both numerical and one-hot columns.
* **`SelectKBest(k=20)`** scores every encoded feature and keeps the 20 with the highest MI score.
* The selector is a **step inside each model's `Pipeline`**, placed after the preprocessing: `preprocessor → feature selector → model`. It is therefore fitted on the **training data only** and then simply applied to the test data, so the test set never influences which features are chosen.
* The target `y` is never part of the inputs, and `duration` stays excluded (Section 3).
* One-hot columns are treated as discrete variables and the scaled numerical columns as continuous ones when MI is calculated (the helper function below does this).
* `K_FEATURES = 20` is a fixed choice (about 40% of the encoded features); it was not tuned on the test set.

In [ ]:
import numpy as np
from scipy.sparse import issparse
from sklearn.base import clone
from sklearn.feature_selection import SelectKBest, mutual_info_classif

K_FEATURES = 20   # number of encoded features to keep

def mutual_info_mixed(X, y):
    """Mutual information with the right feature types: the scaled numerical columns come first
    in the ColumnTransformer output (continuous); the one-hot columns come after (discrete)."""
    X = X.toarray() if issparse(X) else np.asarray(X)
    discrete = np.arange(X.shape[1]) >= len(numeric_features)
    return mutual_info_classif(X, y, discrete_features=discrete, random_state=RANDOM_STATE)

# This selector is added to every model Pipeline in Section 4
feature_selector = SelectKBest(score_func=mutual_info_mixed, k=K_FEATURES)

# ---- Show what it selects: fit on the TRAINING data only (a separate copy, nothing is shared) ----
prep_demo = clone(preprocessor).fit(X_train)
X_train_encoded = prep_demo.transform(X_train)
selector_demo = clone(feature_selector).fit(X_train_encoded, y_train)

def original_name(encoded_name):
    """'cat__job_student' -> 'job', 'num__age' -> 'age'"""
    kind, rest = encoded_name.split("__", 1)
    if kind == "num":
        return rest
    return max((c for c in categorical_features if rest.startswith(c + "_")), key=len)

encoded_names = prep_demo.get_feature_names_out()
feature_scores = pd.DataFrame({
    "Encoded feature": [n.split("__", 1)[1] for n in encoded_names],
    "Original feature": [original_name(n) for n in encoded_names],
    "MI score": selector_demo.scores_,
    "Selected": selector_demo.get_support(),
}).sort_values("MI score", ascending=False).reset_index(drop=True)
feature_scores.index += 1   # rank starts at 1

print(f"Encoded features before selection: {len(encoded_names)} | kept: {int(feature_scores['Selected'].sum())}")
print("\nSelected features (ranked by mutual information):")
display(feature_scores[feature_scores["Selected"]].drop(columns="Selected").round(4))

kept_original = feature_scores.loc[feature_scores["Selected"], "Original feature"].unique().tolist()
dropped_original = [c for c in numeric_features + categorical_features if c not in kept_original]
print("Original features with at least one selected column:", kept_original)
print("Original features with NO selected column          :", dropped_original)

# Bar chart: top-scoring features, selected ones highlighted
top = feature_scores.head(K_FEATURES + 5).iloc[::-1]
colors = ["tab:blue" if s else "lightgrey" for s in top["Selected"]]
plt.figure(figsize=(8, 7))
plt.barh(top["Encoded feature"], top["MI score"], color=colors)
plt.xlabel("Mutual information score")
plt.title(f"Feature selection: top {K_FEATURES + 5} features (blue = selected)")
plt.tight_layout()
plt.show()

**How to read the feature selection results**

* The table lists the selected columns from the most to the least informative. A column such as `month_mar` is one value of the original feature `month`; the *Original feature* column shows where it came from.
* Features with high scores are the ones that best separate customers who subscribed from those who did not. Look at which original features appear most often near the top of the ranking: those are the customer characteristics that carry the most information about subscribing.
* The grey bars are the next-best features that just missed the cut. Features with scores close to 0 add almost no information and are dropped, which makes the models simpler and less likely to learn noise.
* In Section 4 the selector is part of every pipeline, so the four models are trained and evaluated only on the selected features, and the numbers in Sections 5 and 6 are the results **with** feature selection.

## 4. Model training

All four models are trained on the **same training split**, with the same preprocessing and balanced class weights (`class_weight="balanced"`, or the equivalent balanced sample weights for Gradient Boosting, which has no `class_weight` option). Each pipeline is `preprocessor → feature selector (SelectKBest, Section 3.2) → model`. Settings are fixed in advance (not tuned on the test set).

| Model | Idea |
|---|---|
| Logistic Regression | Combines features into a probability using a weighted sum |
| Decision Tree | Asks a series of yes/no questions about the features (depth limited to avoid memorising the data) |
| Random Forest | Many decision trees that vote together |
| Gradient Boosting | Builds small trees one after another; each new tree tries to correct the mistakes of the previous ones |

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                            class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10,
                                            class_weight="balanced", random_state=RANDOM_STATE,
                                            n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                                                    random_state=RANDOM_STATE),
}

# Gradient Boosting has no class_weight option, so we give the rare 'yes' class a larger
# sample weight instead (same idea as class_weight="balanced"; computed from training labels only)
balanced_weights = compute_sample_weight("balanced", y_train)

trained_models = {}
for name, model in models.items():
    # Pipeline = preprocessing + feature selection + model, fitted on the training data only
    pipeline = Pipeline([("preprocessor", preprocessor), ("selector", feature_selector), ("model", model)])
    if name == "Gradient Boosting":
        pipeline.fit(X_train, y_train, model__sample_weight=balanced_weights)
    else:
        pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    print("Trained:", name)

## 5. Model evaluation

Every model is evaluated on the same untouched **test set**. Accuracy, precision, recall and F1 use the **predicted labels**; ROC-AUC uses the **predicted probabilities** of class 1 (`yes`). Precision, recall and F1 are for the `yes` class.

In [ ]:
results = []
predictions = {}   # saved so the graphs in Section 6 can reuse them
for name, pipeline in trained_models.items():
    y_pred = pipeline.predict(X_test)                  # predicted labels (0 or 1)
    y_prob = pipeline.predict_proba(X_test)[:, 1]      # predicted probability of 'yes'
    predictions[name] = (y_pred, y_prob)

    scores = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }
    results.append(scores)

    print(f"{name}")
    for metric, value in scores.items():
        if metric != "Model":
            print(f"   {metric:10s}: {value:.4f}")

## 6. Model comparison

The table is sorted by **ROC-AUC**, our primary selection metric, because it measures how well a model ranks likely subscribers above unlikely ones without depending on one fixed cut-off, and it is not fooled by class imbalance.

In [ ]:
comparison = (pd.DataFrame(results)
              .set_index("Model")
              .sort_values("ROC-AUC", ascending=False)
              .round(4))
display(comparison)

best_model = comparison.index[0]
print(f"Best model by ROC-AUC: {best_model} (ROC-AUC = {comparison.loc[best_model, 'ROC-AUC']:.4f})")
for metric in ["Accuracy", "Precision", "Recall", "F1-score"]:
    print(f"Highest {metric:9s}: {comparison[metric].idxmax()} ({comparison[metric].max():.4f})")

### Graphs: comparing the models

Three graphs only: metric comparison, confusion matrices and ROC curves. All use the **test set**.

In [ ]:
# Graph 1: all five metrics for the four models
ax = comparison.plot(kind="bar", figsize=(10, 5), rot=0, colormap="Set2")
ax.set_title("Model comparison on the test set")
ax.set_xlabel("Model")
ax.set_ylabel("Score (0 to 1)")
ax.set_ylim(0, 1)
ax.legend(title="Metric", loc="upper right", bbox_to_anchor=(1.18, 1))
plt.tight_layout()
plt.show()

**Graph 1** shows each metric side by side (higher is better). Compare bars of the *same colour* across models. Notice that a model can be best on one metric (e.g. accuracy) but not on another (e.g. recall).

In [ ]:
# Graph 2: confusion matrix of each model
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (name, (y_pred, y_prob)) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["no", "yes"],
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
fig.suptitle("Confusion matrices on the test set")
plt.tight_layout()
plt.show()

**Graph 2:** each square counts test customers. Bottom-right = real subscribers found (true positives); bottom-left = real subscribers missed (false negatives); top-right = customers wrongly predicted "yes" (wasted calls); top-left = correct "no" predictions.

In [ ]:
# Graph 3: ROC curves of the four models
fig, ax = plt.subplots(figsize=(6, 5))
for name, (y_pred, y_prob) in predictions.items():
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random guessing")
ax.set_title("ROC curves on the test set")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate (recall)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Graph 3:** the closer a curve is to the top-left corner, the better the model ranks real subscribers above non-subscribers. The dashed diagonal is random guessing; the area under each curve is the ROC-AUC value in the table.

In [ ]:
# Why accuracy alone can mislead: compare with a 'model' that always answers "no"
always_no_accuracy = 1 - y_test.mean()
print(f"Accuracy of always predicting 'no': {always_no_accuracy:.4f}")
print("Best trained-model accuracy        :", comparison["Accuracy"].max())

### What the metrics mean

* **Accuracy:** out of all customers, the share predicted correctly.
* **Precision:** out of customers predicted as "yes", the share who really subscribed (low precision = many wasted calls).
* **Recall:** out of customers who really subscribed, the share the model found (low recall = many missed customers).
* **F1-score:** one number balancing precision and recall.
* **ROC-AUC:** how well the model ranks customers (0.5 = random guessing, 1.0 = perfect).

### Why accuracy alone can be misleading

Only a small share of customers subscribe, so a useless model that always answers "no" already gets high accuracy (printed above) while finding **no** subscribers. Because we use `class_weight="balanced"`, our models deliberately accept some extra wrong "yes" predictions in order to find more real subscribers, which can even lower their accuracy below that baseline. That is why precision, recall, F1 and ROC-AUC are needed alongside accuracy.

*Note: the model that is "best" depends on the goal. ROC-AUC is used here as the single selection metric; if the bank cared most about not missing subscribers, recall would be weighted more.*